# Lab 10: Capstone Integration — Structured Agentic BI Cockpit (JSON Schema + UI)

**Course / Disciplina:** Augmented Analytics & AI-Driven Insights (FIAP MBA)  
**[🇧🇷 PT-BR]** Integrar o contrato determinístico JSON Schema (Lab 08) à interface executiva Streamlit (Lab 09).  
**[🇺🇸 EN]** Unify deterministic JSON Schema contracts (Lab 08) with the reactive Streamlit executive cockpit (Lab 09).

---
### 🚀 1. Dependency Installation & Sample Ingestion / Instalação de Dependências

In [ ]:
!pip install crewai streamlit pyngrok pandas -q
import urllib.request, os
if not os.path.exists('PesquisaClientes.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/rafaelmatsuyama/FIAP-BI-AugmentedAnalytics/main/lab01-cx-analytics/Lab%2001%20-%20PesquisaClientes.csv', 'PesquisaClientes.csv')
    print('✅ Demo dataset downloaded successfully! / Dataset de demonstração baixado com sucesso!')

### 📝 2. Integrated Dashboard Synthesis (`app.py`) / Desenvolvimento do Dashboard Integrado
* 🇧🇷 A célula abaixo grava `app.py` integrando o Agente CrewAI, extração segura de JSON via `re.search` e fallback automático para `PesquisaClientes.csv`.
* 🇺🇸 The cell below writes `app.py`, unifying the CrewAI Agent, regex-hardened JSON extraction (`re.search`), and automatic fallback to `PesquisaClientes.csv`.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os
import json
import re
from crewai import Agent, Task, Crew, Process

# -----------------------------------------------------------------
# 1. UI CONFIGURATION
# -----------------------------------------------------------------
st.set_page_config(page_title="BI Pro Dashboard", layout="wide")
st.title("🏢 Agentic Analytics: Insight-as-a-Service")

with st.sidebar:
    st.header("⚙️ Control Panel / Painel de Controle")
    api_key = st.text_input("🔑 Gemini API Key:", type="password")
    output_lang = st.selectbox("🌐 Output Language / Idioma:", ["Portuguese (Brazil)", "English"])
    st.divider()
    st.info("MBA BI & Analytics - FIAP")

# -----------------------------------------------------------------
# 2. INTEGRATION LOGIC (UPLOAD OR SELF-HEALING DEFAULT)
# -----------------------------------------------------------------
uploaded_file = st.file_uploader("📂 Upload CSV dataset (or leave empty to use default PesquisaClientes.csv):", type="csv")
data_source = uploaded_file if uploaded_file is not None else ('PesquisaClientes.csv' if os.path.exists('PesquisaClientes.csv') else None)

if data_source and api_key:
    df = pd.read_csv(data_source)
    if uploaded_file is None:
        st.caption("ℹ️ Using pre-loaded Colab dataset: `PesquisaClientes.csv` / Usando base padrão do Colab.")
    st.dataframe(df.head(10), use_container_width=True)
    
    if st.button("🚀 Generate Structured Analysis / Gerar Análise Estruturada"):
        os.environ["GEMINI_API_KEY"] = api_key
        os.environ["GOOGLE_API_KEY"] = api_key
        
        with st.spinner("🧠 Squad processing telemetry via JSON Schema... / Processando dados... Aguarde."):
            
            # STEP 1: System Instructions (Executive Tone & ROI Focus)
            instrucoes_sistemicas = f"You are a Senior Strategy Consultant from FIAP's MBA in BI & Analytics. Your tone is pragmatic, ROI-driven, and strictly grounded in the provided data. Write all string values in {output_lang}."
            
            analyst = Agent(
                role='FIAP Strategic BI Consultant',
                goal=f'Synthesize actionable, schema-compliant strategic insights in {output_lang}',
                backstory=instrucoes_sistemicas,
                llm="gemini/gemini-3.8-flash"
            )
            
            # STEP 2: Controlled context sampling (df.head(15)) to prevent token exhaustion
            dataset_contexto = df.head(15).to_csv(index=False)
            
            task = Task(
                description=f'''Analyze the telemetry sample below and synthesize a structured strategic insight (values in {output_lang}):\n\n{dataset_contexto}\n\nReturn MANDATORILY a JSON object matching this exact schema:\n{{\n  "titulo": "string",\n  "metrica_critica": "string",\n  "recomendacao": "string",\n  "impacto_estimado": "Baixo" | "Médio" | "Alto"\n}}\n\nReturn ONLY raw JSON, with no extra markdown or commentary.''',
                expected_output='A valid JSON object with keys: titulo, metrica_critica, recomendacao, impacto_estimado.',
                agent=analyst
            )
            
            crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
            raw_result = crew.kickoff()
            
            # -----------------------------------------------------------------
            # 3. REGEX-HARDENED EXTRACTION & SESSION STATE PERSISTENCE
            # -----------------------------------------------------------------
            try:
                raw_text = str(raw_result.raw if hasattr(raw_result, 'raw') else raw_result)
                json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
                
                if json_match:
                    st.session_state["analise_data"] = json.loads(json_match.group(0))
                    st.session_state["raw_output"] = raw_text
                else:
                    raise ValueError("JSON object not found in agent output / JSON não identificado.")
                    
            except Exception as e:
                st.error(f"❌ Error parsing JSON / Erro ao processar JSON: {e}")
                st.write("**Raw Output / Retorno Bruto:**", raw_result)

    # -----------------------------------------------------------------
    # 4. EXECUTIVE COCKPIT RENDERING
    # -----------------------------------------------------------------
    if "analise_data" in st.session_state:
        data = st.session_state["analise_data"]
        st.success("✅ Decision Intelligence Generated! / Inteligência Gerada com Sucesso!")
        st.divider()
        
        col1, col2 = st.columns(2)
        
        with col1:
            st.metric(label="Insight Title / Título do Insight", value=data.get('titulo', 'N/A'))
            st.write(f"**Critical Metric / Métrica Crítica:** {data.get('metrica_critica', 'N/A')}")
        
        with col2:
            st.metric(label="Estimated Impact / Impacto Estimado", value=data.get('impacto_estimado', 'Médio'))
            st.warning(f"**Recommendation / Recomendação:** {data.get('recomendacao', 'N/A')}")
        
        st.divider()
        st.write("### 🛠️ Raw JSON Contract (IT Audit / Auditoria IT)")
        st.json(data)

elif not api_key:
    st.warning("⚠️ Enter your Gemini API Key in the sidebar to activate the Agent Squad. / Insira sua Gemini API Key no menu lateral.")

### 🌐 3. Publication & Secure Tunneling (PyNgrok) / Publicação & Túnel
* 🇧🇷 Insira seu Token do Ngrok abaixo e execute a célula para abrir o link público do Dashboard.
* 🇺🇸 Enter your Ngrok AuthToken below and run the cell to expose the public Dashboard URL.

In [ ]:
# @title Ngrok Tunnel Configuration / Configuração do Túnel
NGROK_TOKEN = "" # @param {type:"string"}

from pyngrok import ngrok
import os

if NGROK_TOKEN:
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print(f"🔗 LIVE EXECUTIVE DASHBOARD / DASHBOARD PROFISSIONAL ATIVO EM: {public_url}")
    print("--- ")
    !streamlit run app.py --server.port 8501
else:
    print("❌ ERROR: Please provide your NGROK_TOKEN above. / ERRO: Insira o NGROK_TOKEN para gerar o link.")